# Introduction - Customer Churn Prediction notebook
In this notebook, we illustrate how you can train a model for Churn Prediction using PySpark. After training the model, you step through the instructions to deploy the model using Watson Machine Learning.

## Package installation

In [1]:
try:
    from pyspark.sql import SparkSession
except:
    print('Error: Spark runtime is missing. If you are using Watson Studio change the notebook runtime to Spark.')
    raise 
    
!pip list | grep ibm-watson-openscale     

ibm-watson-openscale          3.0.37


In [2]:
# install required Python modules

!pip install --upgrade ibm-watsonx-ai --user | tail -n 1
# !pip install --upgrade "ibm-watson-openscale~=3.0.34" --no-cache --user | tail -n 1


[notice] A new release of pip is available: 25.0.1 -> 25.3
[notice] To update, run: pip install --upgrade pip


# Model building and deployment <a name="model"></a>

In this section you will learn how to train Spark MLLib model and next deploy it as web-service using Watson Machine Learning service.

## Load the training data 

- Click in the next cell to insert the code to import the training dataset.
- Click the **Code snippet </>** icon in the top right, find the data set you'd like to import (for example, CUSTOMER_DATA_ready or customer_training_data.csv) into this notebook and click **Insert to code** drop down and select **pandas DataFrame**

In [3]:
import itc_utils.flight_service as itcfs

nb_data_request = {
    'data_name': """customer_training_data.csv""",
    'interaction_properties': {
        #'row_limit': 500,
        'infer_schema': 'true',
        'infer_as_varchar': 'false'
    }
}
flight_descriptor = itcfs.get_flight_descriptor(nb_data_request=nb_data_request)

flightClient = itcfs.get_flight_client()
flightInfo = flightClient.get_flight_info(flight_descriptor)

df_0 = itcfs.read_pandas_and_concat(flightClient, flightInfo, timeout=240)
df_0.head(10)


,ID,LONGDISTANCE,INTERNATIONAL,LOCAL,DROPPED,PAYMETHOD,LOCALBILLTYPE,LONGDISTANCEBILLTYPE,USAGE,RATEPLAN,GENDER,STATUS,CHILDREN,ESTINCOME,CAROWNER,AGE,CHURN
0,1,23,0,206,0,CC,Budget,Intnl_discount,229,3,F,S,1,38000.00,N,24.393333,T
1,1004,28,0,60,0,Auto,FreeLocal,Standard,89,4,F,M,1,8073.11,N,46.000000,F
2,1005,24,0,5,0,CH,Budget,Standard,29,4,M,M,0,95448.60,Y,53.680000,F
3,1006,28,0,97,0,CC,FreeLocal,Standard,125,1,M,S,1,24141.50,Y,17.006667,T
4,1008,0,0,4,2,CC,Budget,Standard,4,2,M,S,1,31952.00,N,34.266667,F
5,1009,29,0,9,0,CC,Budget,Intnl_discount,38,2,M,S,2,72084.70,N,55.640000,F
6,1010,13,0,40,0,CC,Budget,Standard,53,4,F,S,0,42760.50,N,47.000000,F
7,1016,16,0,114,0,CH,Budget,Standard,130,1,M,M,1,71472.90,N,41.913333,T
8,1017,7,0,6,0,CC,Budget,Standard,13,3,F,M,0,95405.70,N,48.000000,F
9,1018,21,0,87,0,CC,Budget,Standard,108,1,F,S,0,95786.80,Y,52.646667,F


In [4]:
# Create a PySpark DataFrame from the pandas DataFrame
from pyspark.sql import SparkSession
import pandas as pd

import json
# Provide the name of the pandas DataFrame from the previous cell (should be of the format df_data_<some_number>)
pandasDFname=df_0
spark = SparkSession.builder.getOrCreate()
sparkDF=spark.createDataFrame(pandasDFname)
sparkDF.head()

Row(ID=1, LONGDISTANCE=23, INTERNATIONAL=0, LOCAL=206, DROPPED=0, PAYMETHOD='CC', LOCALBILLTYPE='Budget', LONGDISTANCEBILLTYPE='Intnl_discount', USAGE=229, RATEPLAN=3, GENDER='F', STATUS='S', CHILDREN=1, ESTINCOME=38000.0, CAROWNER='N', AGE=24.393333, CHURN='T')

## Explore data

In [5]:
sparkDF.printSchema()

root
 |-- ID: long (nullable = true)
 |-- LONGDISTANCE: long (nullable = true)
 |-- INTERNATIONAL: long (nullable = true)
 |-- LOCAL: long (nullable = true)
 |-- DROPPED: long (nullable = true)
 |-- PAYMETHOD: string (nullable = true)
 |-- LOCALBILLTYPE: string (nullable = true)
 |-- LONGDISTANCEBILLTYPE: string (nullable = true)
 |-- USAGE: long (nullable = true)
 |-- RATEPLAN: long (nullable = true)
 |-- GENDER: string (nullable = true)
 |-- STATUS: string (nullable = true)
 |-- CHILDREN: long (nullable = true)
 |-- ESTINCOME: double (nullable = true)
 |-- CAROWNER: string (nullable = true)
 |-- AGE: double (nullable = true)
 |-- CHURN: string (nullable = true)



In [6]:
print("Number of records: " + str(sparkDF.count()))

Number of records: 1415


## Preprocessing

In [7]:
from pyspark.sql.functions import count, when, col, sum
print(sparkDF.select([count(when(col(c).isNull(), c)).alias(c) for c in sparkDF.columns]))
has_nulls = sparkDF.select([count(when(col(c).isNull(), c)).alias(c) for c in sparkDF.columns]).collect()[0]
if any(value > 0 for value in has_nulls.asDict().values()):
    print("DataFrame contains null values.")
else:
    print("DataFrame does not contain null values.")


DataFrame[ID: bigint, LONGDISTANCE: bigint, INTERNATIONAL: bigint, LOCAL: bigint, DROPPED: bigint, PAYMETHOD: bigint, LOCALBILLTYPE: bigint, LONGDISTANCEBILLTYPE: bigint, USAGE: bigint, RATEPLAN: bigint, GENDER: bigint, STATUS: bigint, CHILDREN: bigint, ESTINCOME: bigint, CAROWNER: bigint, AGE: bigint, CHURN: bigint]
DataFrame does not contain null values.


In [8]:
# Check for missing values
sparkDF.select(*(sum(col(c).isNull().cast("int")).alias(c) for c in sparkDF.columns)).show()

+---+------------+-------------+-----+-------+---------+-------------+--------------------+-----+--------+------+------+--------+---------+--------+---+-----+
| ID|LONGDISTANCE|INTERNATIONAL|LOCAL|DROPPED|PAYMETHOD|LOCALBILLTYPE|LONGDISTANCEBILLTYPE|USAGE|RATEPLAN|GENDER|STATUS|CHILDREN|ESTINCOME|CAROWNER|AGE|CHURN|
+---+------------+-------------+-----+-------+---------+-------------+--------------------+-----+--------+------+------+--------+---------+--------+---+-----+
|  0|           0|            0|    0|      0|        0|            0|                   0|    0|       0|     0|     0|       0|        0|       0|  0|    0|
+---+------------+-------------+-----+-------+---------+-------------+--------------------+-----+--------+------+------+--------+---------+--------+---+-----+



In [9]:
duplicate_count = sparkDF.groupBy(sparkDF.columns).count().filter("count > 1").count()
print(f"Number of duplicate rows: {duplicate_count}")

Number of duplicate rows: 0


## Create a model

In [10]:
spark_df = sparkDF
# Split the labeled data into a training set and a test set
(train_data, test_data) = spark_df.randomSplit([0.8, 0.2], 24)

# Provide a target name for your churn model
MODEL_NAME = "Churn Model"
# Provide a target name for your churn model deployment
DEPLOYMENT_NAME = "Churn Deployment"

print("Number of records for training: " + str(train_data.count()))
print("Number of records for evaluation: " + str(test_data.count()))

Number of records for training: 1158
Number of records for evaluation: 257


The code below creates a Random Forest Classifier with Spark, setting up string indexers for the categorical features and the label column. Finally, this notebook creates a pipeline including the indexers and the model, and does an initial Area Under ROC evaluation of the model.

In [11]:
from pyspark.ml.feature import OneHotEncoder, StringIndexer, IndexToString, VectorAssembler
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml import Pipeline, Model
from pyspark.ml.feature import SQLTransformer

features = [x for x in spark_df.columns if x != 'CHURN']
# Specify the categorical features
categorical_features = ['PAYMETHOD', 'LOCALBILLTYPE', 'LONGDISTANCEBILLTYPE', 'GENDER', 'STATUS', 'CAROWNER']
# Index the categorical feature so each string value is replaced with an integer
categorical_num_features = [x + '_IX' for x in categorical_features]
si_list = [StringIndexer(inputCol=x, outputCol=y) for x, y in zip(categorical_features, categorical_num_features)]
va_features = VectorAssembler(inputCols=categorical_num_features + [x for x in features if x not in categorical_features], outputCol="features")

In [12]:
# Index the label column
si_label = StringIndexer(inputCol="CHURN", outputCol="label").fit(spark_df)
label_converter = IndexToString(inputCol="prediction", outputCol="predictedLabel", labels=si_label.labels)

In [13]:
from pyspark.ml.classification import RandomForestClassifier
# train a Random Forect Classifier
classifier = RandomForestClassifier(featuresCol="features")
pipeline = Pipeline(stages= si_list + [si_label, va_features, classifier, label_converter])

model = pipeline.fit(train_data)

In [14]:
predictions = model.transform(test_data)
evaluatorDT = BinaryClassificationEvaluator(rawPredictionCol="prediction",  metricName='areaUnderROC')
area_under_curve = evaluatorDT.evaluate(predictions)

evaluatorDT = BinaryClassificationEvaluator(rawPredictionCol="prediction",  metricName='areaUnderPR')
area_under_PR = evaluatorDT.evaluate(predictions)
#default evaluation is areaUnderROC
print("areaUnderROC = %g" % area_under_curve, "areaUnderPR = %g" % area_under_PR)

areaUnderROC = 0.908512 areaUnderPR = 0.82466


In [15]:
# extra code: evaluate more metrics by exporting them into pandas and numpy
from sklearn.metrics import classification_report
y_pred = predictions.toPandas()['prediction']
y_pred = ['T' if pred == 1.0 else 'F' for pred in y_pred]
y_test = test_data.toPandas()['CHURN']
print(classification_report(y_test, y_pred, target_names=['T', 'F']))

              precision    recall  f1-score   support

           T       0.95      0.90      0.92       162
           F       0.84      0.92      0.88        95

    accuracy                           0.91       257
   macro avg       0.90      0.91      0.90       257
weighted avg       0.91      0.91      0.91       257



## watsonx.ai connection

Authenticate the watsonx.ai Runtime service on IBM Cloud. You need to provide platform api_key and instance location.

You can find your api_key by clicking the **icon at top right > Profile and Settings > API Key > Generate a new key.**

In [16]:
import getpass

api_key = getpass.getpass("Please enter your api key (press enter): ")

Please enter your api key (press enter):  ········


In [22]:
credentials = {
    'username': 'cpadmin',
    'apikey': api_key,
    'url' : 'https://cpd-cpd-instance.apps.691c390807fea3b75366233a.am1.techzone.ibm.com/',
    'instance_id': 'openshift'
}

In [23]:
print(credentials)

{'username': 'cpadmin', 'apikey': 'YA7hNpcPOIZkaQxeeroVlAuGp6aiSwnsLhs66aD8', 'url': 'https://cpd-cpd-instance.apps.691c390807fea3b75366233a.am1.techzone.ibm.com/', 'instance_id': 'openshift'}


In [24]:
from ibm_watsonx_ai import APIClient
client = APIClient(credentials)

In [26]:
# List existing deployment space
client.spaces.list(limit=10)

,ID,NAME,CREATED
0,4ec09120-3d15-430d-8d3e-8ffb6023ab86,Test,2025-11-25T09:28:03.980Z


## Publish the model

In [27]:
def getSpaceIDwml(wml_client,space_name):
    spaces = wml_client.spaces.get_details()['resources'];
    try:
        spaceList = next(item for item in spaces if item['entity']['name']==space_name)
        spaceID = spaceList['metadata']['id']
    except:
        spaceID = -1
    return spaceID

In [42]:
def createSpacewml(wml_client,space_name):
    spaces = wml_client.spaces.get_details()['resources'];
    for space in spaces:
        if space['entity']['name'] ==space_name:
            print("Deployment space with name",space_name,"already exists . .")
            return space['metadata']['id']
    print("\nCreating a new deployment space -",space_name)
    # create the space
    space_meta_data = {
        wml_client.spaces.ConfigurationMetaNames.NAME : space_name
    }

    stored_space_details = wml_client.spaces.store(space_meta_data)
    space_id = stored_space_details['metadata']['id']
    i=0
    while(True):
        stored_space_details=wml_client.spaces.get_details(space_id)
        status=stored_space_details['entity']['status']['state']
        print("i: ", i, " status: ", status)
        if status == 'active':
            break
        time.sleep(1)
        i = i+1
    return space_id

In [34]:
# Associate Watson Machine Learning with a specific space

space_name = 'Test' #Replace any name ò choice here
space_id=getSpaceIDwml(client,space_name)
if space_id == -1:
    space_id = createSpacewml(client,space_name)
print('space id: ', space_id)
client.set.default_space(space_id)


space id:  4ec09120-3d15-430d-8d3e-8ffb6023ab86


In [30]:
def deleteExistingModelsSameName(client,model_name):
    stored_models=client.repository.get_model_details()
    stored_models_details = stored_models['resources']
    for m in stored_models_details:
        m_name = m['metadata']['name']
        if m_name == model_name:
            model_id = m['metadata']['id']
            print("Deleteing model with id: ", model_id, " and name: ", m_name)
            client.repository.delete(model_id)
    return 'Success'

In [31]:
def deleteExistingDeploymentsSameName(client,deployment_name):
    stored_deployments=client.deployments.get_details()
    stored_deployment_details = stored_deployments['resources']
    for d in stored_deployment_details:
        d_name = d['metadata']['name']
        if d_name == deployment_name:
            deployment_id = d['metadata']['id']
            print("Deleteing deployment with id: ", deployment_id, " and name: ", d_name)
            client.deployments.delete(deployment_id)
    return 'Success'

Previous versions of the model are removed so that the notebook can be run again, resetting all data for another demo.

In [32]:
# Delete Existing Deployments with same name
deleteExistingDeploymentsSameName(client,DEPLOYMENT_NAME)

'Success'

In [33]:
# Delete existing models with same name
deleteExistingModelsSameName(client,MODEL_NAME)

'Success'

In [29]:
software_spec_uid = client.software_specifications.get_id_by_name("spark-mllib_3.5")
print("Software Specification ID: {}".format(software_spec_uid))
model_props = {
        client._models.ConfigurationMetaNames.NAME:"{}".format(MODEL_NAME),
        client._models.ConfigurationMetaNames.TYPE: "mllib_3.5",
        client._models.ConfigurationMetaNames.SOFTWARE_SPEC_UID: software_spec_uid,
        #wml_client._models.ConfigurationMetaNames.TRAINING_DATA_REFERENCES: training_data_references,
        client._models.ConfigurationMetaNames.LABEL_FIELD: "CHURN",
    }

Software Specification ID: e8cd7001-fd05-5eea-b697-0a3bff9cf51f


In [38]:
print("Storing model ...")
published_model_details = client.repository.store_model(
    model=model, 
    meta_props=model_props, 
    training_data=train_data, 
    pipeline=pipeline)

model_uid = client.repository.get_model_id(published_model_details)
print("Done")
print("Model ID: {}".format(model_uid))

Storing model ...
Done
Model ID: 3beadbd7-e2cf-4aed-a0a7-e44cde56cc4f


In [39]:
client.repository.list_models()

,ID,NAME,CREATED,TYPE,SPEC_STATE,SPEC_REPLACEMENT
0,3beadbd7-e2cf-4aed-a0a7-e44cde56cc4f,Churn Model,2025-12-10T07:43:34Z,mllib_3.5,supported,
1,51a6d98b-e98d-47cd-aa39-edbe8bd0b57f,pkl test 1.6 sklearn,2025-11-25T10:48:49Z,scikit-learn_1.6,supported,
2,5c0d6758-f505-418d-903d-969ab03d2dff,pkl test,2025-11-25T09:28:14Z,scikit-learn_1.3,supported,


## Deploy the model

The next section of the notebook deploys the model as a RESTful web service in Watson Machine Learning. The deployed model will have a scoring URL you can use to send data to the model for predictions.

In [41]:
deployment_details = client.deployments.create(
    model_uid, 
    meta_props={
        client.deployments.ConfigurationMetaNames.NAME: "{}".format(DEPLOYMENT_NAME),
        client.deployments.ConfigurationMetaNames.ONLINE: {}
    }
)




######################################################################################

Synchronous deployment creation for id: '3beadbd7-e2cf-4aed-a0a7-e44cde56cc4f' started

######################################################################################


initializing
Note: online_url is deprecated and will be removed in a future release. Use serving_urls instead.
.................
ready


-----------------------------------------------------------------------------------------------
Successfully finished deployment creation, deployment_id='68a9d773-1d20-42c5-8311-ebd6af006ea9'
-----------------------------------------------------------------------------------------------




NameError: name 'wml_client' is not defined

In [42]:
scoring_url = client.deployments.get_scoring_href(deployment_details)
deployment_uid=client.deployments.get_uid(deployment_details)

print("Scoring URL:" + scoring_url)
print("Model id: {}".format(model_uid))
print("Deployment id: {}".format(deployment_uid))

Scoring URL:https://cpd-cpd-instance.apps.691c390807fea3b75366233a.am1.techzone.ibm.com/ml/v4/deployments/68a9d773-1d20-42c5-8311-ebd6af006ea9/predictions
Model id: 3beadbd7-e2cf-4aed-a0a7-e44cde56cc4f
Deployment id: 68a9d773-1d20-42c5-8311-ebd6af006ea9


In [43]:
client.deployments.list()

,ID,NAME,STATE,CREATED,ARTIFACT_TYPE,SPEC_STATE,SPEC_REPLACEMENT
0,68a9d773-1d20-42c5-8311-ebd6af006ea9,Churn Deployment,ready,2025-12-10T07:44:20.217Z,model,supported,
1,c101513b-6879-45d8-be52-75a491b265f2,WOS-INTERNAL-deb56bda-6308-4b5b-bfe3-baa67f0e8f5b,ready,2025-11-28T04:55:04.292Z,model,supported,
2,deb56bda-6308-4b5b-bfe3-baa67f0e8f5b,churn detector 1,ready,2025-11-25T10:49:03.988Z,model,supported,


## Sample scoring

In [44]:
fields = ["ID","LONGDISTANCE","INTERNATIONAL","LOCAL","DROPPED","PAYMETHOD","LOCALBILLTYPE","LONGDISTANCEBILLTYPE","USAGE",\
            "RATEPLAN","GENDER","STATUS","CHILDREN","ESTINCOME","CAROWNER","AGE"]
values = [[1,28,0,60,0,"Auto","FreeLocal","Standard",89,4,"F","M",1,23000,"N",45]]
scoring_payload = {"input_data": [{"fields": fields, "values": values}]}

In [81]:
scoring_response = client.deployments.score(deployment_uid, scoring_payload)
scoring_response

AttributeError: module 'ibm_watson_openscale.client' has no attribute 'deployments'